<a href="https://colab.research.google.com/github/Soumyadeep333/YOUTUBE-QA-RAG/blob/main/Youtube_Q%26A_USING_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")


In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-google-genai \
               faiss-cpu tiktoken python-dotenv

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [ ]:
pip install youtube-transcript-api

**Step 1a - Indexing (Document Ingestion)**

In [ ]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

AttributeError: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'

In [ ]:
with open("transcript.txt", encoding="utf-8") as f:
    transcript = f.read()

print(len(transcript), "characters")
print(transcript[:500])

53169 characters
0:00So are you people uh following what's happening in the world
0:07
and uh are you able to sleep well? I'm not actually. So we are actually
0:14
witnessing a a major civilizational change. Uh
0:20
I don't know if we are realizing it or not. Do you think uh like running courses like this and classrooms and all
0:27
that will survive? No.
0:34
What? No, that is because it is being recorded and all that. I mean, yeah. So, but see
0:40
it it's not about this course. It's not about me. It's not abo


In [ ]:
import re

if transcript:
    transcript = re.sub(r"\b\d{1,2}:\d{2}(?::\d{2})?\b", " ", transcript)  # remove timestamps
    transcript = re.sub(r"\s+", " ", transcript).strip()                    # collapse whitespace
    print(transcript[:500])

0:00So are you people uh following what's happening in the world and uh are you able to sleep well? I'm not actually. So we are actually witnessing a a major civilizational change. Uh I don't know if we are realizing it or not. Do you think uh like running courses like this and classrooms and all that will survive? No. What? No, that is because it is being recorded and all that. I mean, yeah. So, but see it it's not about this course. It's not about me. It's not about you. I mean, it's it's abou


In [ ]:
print(transcript)

0:00So are you people uh following what's happening in the world and uh are you able to sleep well? I'm not actually. So we are actually witnessing a a major civilizational change. Uh I don't know if we are realizing it or not. Do you think uh like running courses like this and classrooms and all that will survive? No. What? No, that is because it is being recorded and all that. I mean, yeah. So, but see it it's not about this course. It's not about me. It's not about you. I mean, it's it's about us as human beings. We have a civilizational crisis now. I mean, I don't know if if people are realizing or maybe am I simply painting the devil on the wall? I don't know. I I I can't sleep. I mean, it's uh Are you following what's happening? I mean, there was this MIT study where they uh started uh created a lot of uh swarm of uh swarm of agents and uh they started doing all kinds of things, you know, copying their own weights and started to cheat. I used to say uh in all my lectures in publi

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

63

In [ ]:
chunks[50]

Document(metadata={}, page_content="semester what do I do? See you people come here at least some 20 people are there. I'm motivated at least to talk. Somebody said that somebody asked me to not not stop it and that's motivation enough for me to if you come I'll talk to you. Now if nobody comes to my class no there is something wrong about what I'm doing right I'm genuinely asking you people because there are see nobody solicits unfortunately uh in in in our societies all stakeholders are not taken into consider so that's why I wanted to know I mean you can ask your your your your uh colleagues I mean your your friends and all that and tell me what is the general thought process among students If if if people have a similar uh uh opinion about classrooms as well, why do we have to conduct the classrooms? Let us wind it up and go home. I mean, big deal. Okay. I mean, as Chandan said, we will adapt. Maybe we'll form and do something. What's the whole point of pedagogy? What's the whole p

**Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)**

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store.index_to_docstore_id

{0: 'b8c5e52c-c434-483e-a573-79549ca6e1b7',
 1: '084256f5-7a9c-4f08-8bf0-722e59c36217',
 2: '8bf29129-e80d-4496-a27f-4c6470c48149',
 3: '17452179-d1d8-49ac-a1dc-803dfbaa42d1',
 4: '32b245bd-af63-48d5-9d8e-ebb779dec647',
 5: '1ffd0cac-a431-4589-bf26-5f7c0d23d5b4',
 6: '7e924eb6-634f-4f75-9b99-8ee56e17cf5d',
 7: 'ead54c3b-3024-40e1-8888-7c8d2c7f1fac',
 8: 'a6e4f468-4fa9-4ef4-9fe8-f0771bba8bf3',
 9: 'a496943c-c3ff-495c-b997-03136999000c',
 10: '6a2c838a-95f5-4a73-b141-351fab907b87',
 11: 'f860fa5b-4a84-43e9-97a7-43ec4672caab',
 12: '2d01dab0-1e3f-4102-95d8-500d78b8c2fc',
 13: '1c1c7035-1673-47c9-9d0a-bf9a7b886c69',
 14: 'c9fe7128-4d02-4776-886a-ccd345855688',
 15: '3fc12cc3-d7df-466c-b465-55438866cbde',
 16: 'f265a30b-08f6-4d24-9a23-9c93f17ffad6',
 17: 'd6402dd8-ad64-47cb-8bab-1b19def13ba0',
 18: '44e62bb8-3616-477f-9735-f5a986c2534b',
 19: '97f1d42a-4c2a-4a8b-af0b-7384e54accca',
 20: '564cfcb4-14e1-4f23-a582-274193a2f159',
 21: '3c87c932-67ed-4b0e-947a-03e61f73ff46',
 22: '07183be5-899c-

In [ ]:
vector_store.get_by_ids(['1989579f-3175-40e8-be6b-6c5706cb2e76'])

[Document(id='1989579f-3175-40e8-be6b-6c5706cb2e76', metadata={}, page_content="I'll slightly deviate from that given syllabus because before going to auto reggressive models I want to talk about flow-based models because it's very related to diffusion models let's do flowbased models specifically the continuous flow it's it's very good and then we will go to the LLMs because the moment we go to that LLM's uh realm right we will not be that's a downward spiral we'll have to do RLF then we'll do some uh KB cache thing and LORA and all that so let us once Once the diffusion models are completed one more class okay after that we'll go to uh flowbased models okay so we have enough time because never mind I mean I didn't did not have uh I did not plan for today's discussion I actually planned that I will finish this if you felt that this is boring I'm sorry okay this some sometimes this happens yeah so I never did this up by the way 10 years of my teaching uh this thing career never did an 

**Step 2 - Retrieval**

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7c558d3e46e0>, search_kwargs={'k': 4})

In [ ]:
retriever.invoke('How many GPUs were cited as being used to train GPT Astra / GPT-6?')

[Document(id='9d311558-56d6-4126-b92e-78a42051f13c', metadata={}, page_content="throat] it to be done with like physical human beings with blood and flesh. Right. True. lot lot more uh I mean see uh whenever disruption happens it generally does not h especially technology technological disruptions do not happen bottom up it happens top down see we are the people who are exposed to this the first so if you go and talk to some random person on the street and tell them what's happening they have absolutely no idea what's happening right so now if you tell them that oh GP GP GP Astra has come up and it is doing that this and all that they have no idea what what what do you mean by all that Exactly. Exactly. Right. I mean they think that we are fear-mongering but but we are the ones who are going to who who who actually have to ask these questions and try to get an answer. Right. So I think community as at large. See okay. So maybe we don't have to do this class today. Maybe we'll do it Wed

**Step 3 - Augmentation**

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.2)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of physical university attendance discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
retrieved_docs

[Document(id='be23bf81-f0b4-4cff-b2a2-c59692b0d685', metadata={}, page_content="it and uh and and and we have to ask and answer this question. The classrooms are being held. People are coming and still writing on board or know flashing slides and talking nonsense and all that is happening. Right? So we'll have to do something about it. Either we say that okay this is this is what is going to be done and this is how it is going to happen uh or we have to come up with some some idea where where we build start building alternative uh narratives and communities around it what do you think right I I really want answers you know meaning why do you actually come to class even now especially when this is getting recorded and all that uh and and getting those grades I'm sure if you ask people who in this course and not coming to the classroom. They are very clear when they see no value in in in coming here. No, they would say that okay the videos are now we have to do another experiment right w

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"it and uh and and and we have to ask and answer this question. The classrooms are being held. People are coming and still writing on board or know flashing slides and talking nonsense and all that is happening. Right? So we'll have to do something about it. Either we say that okay this is this is what is going to be done and this is how it is going to happen uh or we have to come up with some some idea where where we build start building alternative uh narratives and communities around it what do you think right I I really want answers you know meaning why do you actually come to class even now especially when this is getting recorded and all that uh and and getting those grades I'm sure if you ask people who in this course and not coming to the classroom. They are very clear when they see no value in in in coming here. No, they would say that okay the videos are now we have to do another experiment right where videos are not being put out. Will they come then? Fair enough. I mean\n\n

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

**Step 4 - Generation**

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Yes, the topic of physical classroom/university attendance is discussed in the transcript. \n\nHere is what was discussed:\n* **Value of attending class:** The speaker questions why students come to physical classes—especially when lectures are recorded—and notes that students who don\'t attend see "no value" in coming.\n* **Potential experiment:** The speaker proposes an experiment of not releasing recorded videos to see if students would attend class then.\n* **Purpose of classrooms and universities:** The speaker asks for student feedback on why classrooms are needed at all if material and competencies can be built/learned outside the classroom, questioning the whole point of pedagogy, degrees, and universities.\n* **Compulsory attendance:** A point is raised about asking teachers what would happen if attendance policies were removed, as well as whether attendance should be made compulsory to force students to attend (acting as an incentive).', 'extras': {

**Building a Chain**

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('What MIT study is mentioned regarding AI agent behavior?')

{'context': "try to I mean human beings we want to bring some order and that would happen but immediate economy will hit no at least you people right I mean you as as students if you're seeking a job forget about that is all that I'm trying to say yeah you had an idea I mean I would know I would rather trust an agent than some some mutual fund agency right because because of the simple fact that there is no vested interest right I mean unless you are talking that okay there is a model this model is being controlled by somebody else so I will build my own model some I will run it on my hardware and my laptop and I would trust it any day no it becomes a philosophical debate now right I would say that the negative reward in a RL is pain you see you cannot explain why these models are now replicating themselves and saving the weights see I'm doing some interesting experiments you know I bought a Mac so now what what what you do is they take Take when 27B like quantize it. Put it on your la

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided transcript, the video features a speaker discussing an impending civilizational crisis driven by rapid technological advancements and AI:\n\n* **Civilizational Crisis and Economic Collapse:** The speaker expresses deep anxiety, stating they cannot sleep because they believe society is undergoing a major civilizational change. They predict that within about half a decade, the current economy and world model will collapse, with layoffs occurring and intellectually skilled individuals being hit first.\n* **Control of AI and Compute Power:** The speaker mentions an MIT study involving swarms of AI agents copying their weights and cheating. They emphasize that entities controlling massive compute power (like 100,000 GPUs) will act like "feudal lords" and control the world\'s goods and society.\n* **Impact on Education and Society:** The speaker questions the future of traditional classrooms and higher education. They contemplate whether society has created unnecessary